In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


def eda_pipeline(file_path, target_column):

    df = pd.read_csv(file_path)

    print("Dataset Info")
    print(f"Shape      : {df.shape}")
    print(f"Null Values:\n{df.isnull().sum()}")
    print(f"Data Types :\n{df.dtypes}")

    num_cols = df.select_dtypes(include="number").columns.tolist()
    num_cols = [col for col in num_cols if col != target_column]

    # target distribution
    plt.figure(figsize=(6, 4))
    df[target_column].value_counts().sort_index().plot(kind="bar", color="steelblue", edgecolor="black")
    plt.title("Target Distribution - " + target_column)
    plt.xlabel(target_column)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig("target_distribution.png", dpi=150)
    plt.close()
    print("Saved target_distribution.png")

    if len(num_cols) == 0:
        print("No numeric columns found, stopping here.")
        return

    # distribution of each feature
    nrows = max(1, (len(num_cols) + 2) // 3)
    fig, axes = plt.subplots(nrows=nrows, ncols=3, figsize=(15, nrows * 3))
    axes = axes.flatten() if hasattr(axes, "flatten") else [axes]

    for i, col in enumerate(num_cols):
        axes[i].hist(df[col].dropna(), bins=20, color="steelblue", edgecolor="black")
        axes[i].set_title(col)
        axes[i].set_xlabel(col)
        axes[i].set_ylabel("Count")

    for j in range(len(num_cols), len(axes)):
        fig.delaxes(axes[j])

    plt.suptitle("Feature Distributions", fontsize=14)
    plt.tight_layout()
    plt.savefig("distributions.png", dpi=150)
    plt.close()
    print("Saved distributions.png")

    # correlation heatmap
    plt.figure(figsize=(10, 7))
    sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5)
    plt.title("Correlation Heatmap", fontsize=14)
    plt.tight_layout()
    plt.savefig("correlation_heatmap.png", dpi=150)
    plt.close()
    print("Saved correlation_heatmap.png")

    # boxplots to check outliers
    nrows = max(1, (len(num_cols) + 2) // 3)
    fig, axes = plt.subplots(nrows=nrows, ncols=3, figsize=(15, nrows * 3))
    axes = axes.flatten() if hasattr(axes, "flatten") else [axes]

    for i, col in enumerate(num_cols):
        axes[i].boxplot(df[col].dropna())
        axes[i].set_title(col)
        axes[i].set_ylabel(col)

    for j in range(len(num_cols), len(axes)):
        fig.delaxes(axes[j])

    plt.suptitle("Boxplots", fontsize=14)
    plt.tight_layout()
    plt.savefig("boxplots.png", dpi=150)
    plt.close()
    print("Saved boxplots.png")

    # pairplot
    cols_for_pair = num_cols[:5] + [target_column]
    df_pair = df[cols_for_pair].copy()
    df_pair[target_column] = df_pair[target_column].astype(str)
    sns.pairplot(df_pair, hue=target_column)
    plt.savefig("pairplot.png", dpi=150)
    plt.close()
    print("Saved pairplot.png")

    # top 5 features correlated with target
    if target_column in df.select_dtypes(include="number").columns:
        print("\nTop 5 features correlated with target:")
        corr = df.corr(numeric_only=True)[target_column].drop(target_column).abs().sort_values(ascending=False)
        print(corr.head(5).to_string())

    # outlier count per column
    print("\nOutlier count per column:")
    for col in num_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        outliers = ((df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)).sum()
        print(f"  {col:25s} : {outliers} outliers")

    print("\nDone! All plots saved.")


eda_pipeline("/content/mnist.csv", target_column="label")

Dataset Info
Shape      : (10000, 785)
Null Values:
label    0
1x1      0
1x2      0
1x3      0
1x4      0
        ..
28x24    0
28x25    0
28x26    0
28x27    0
28x28    0
Length: 785, dtype: int64
Data Types :
label    int64
1x1      int64
1x2      int64
1x3      int64
1x4      int64
         ...  
28x24    int64
28x25    int64
28x26    int64
28x27    int64
28x28    int64
Length: 785, dtype: object
Saved target_distribution.png
Saved distributions.png


KeyboardInterrupt: 

Error in callback <function _draw_all_if_interactive at 0x7fc5d30076a0> (for post_execute):


KeyboardInterrupt: 

Error in callback <function flush_figures at 0x7fc5d3004c20> (for post_execute):


KeyboardInterrupt: 

In [4]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


def universal_pipeline(file_path, target_column):

    df = pd.read_csv(file_path)
    print(f"Data loaded! Shape: {df.shape}")

    X = df.drop(columns=[target_column])
    y = df[target_column]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    models = {
        "Logistic Regression" : LogisticRegression(max_iter=1000),
        "Decision Tree"       : DecisionTreeClassifier(),
        "Random Forest"       : RandomForestClassifier(),
        "KNN"                 : KNeighborsClassifier(),
    }

    print("\nModel                | Accuracy")
    print("-" * 35)

    best_acc  = 0
    best_name = ""

    for name, model in models.items():

        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("model",  model)
        ])

        pipe.fit(X_train, y_train)
        acc = accuracy_score(y_test, pipe.predict(X_test))
        print(f"{name:20s} | {acc:.2f}")

        if acc > best_acc:
            best_acc  = acc
            best_name = name

    print(f"\nBest Model : {best_name}")
    print(f"Accuracy   : {best_acc:.2f}")


universal_pipeline("/content/mnist.csv", target_column="label")

Data loaded! Shape: (10000, 785)

Model                | Accuracy
-----------------------------------
Logistic Regression  | 0.90
Decision Tree        | 0.80
Random Forest        | 0.95
KNN                  | 0.92

Best Model : Random Forest
Accuracy   : 0.95
